## 🧠 10. Foundry Memory Store — Persistent User Preferences

**Goal**: Demonstrate how to use Microsoft Foundry's Memory Store to persist user preferences across conversations. The agent remembers what the user tells it in one session and recalls it automatically in the next.

**Scenario**: A personal shopping assistant that remembers:
- Clothing size and style preferences
- Favorite brands and colors
- Budget constraints
- Past purchase context

**Key concepts demonstrated**:
- Creating and configuring a Memory Store (via `azure.ai.projects`)
- Attaching the `MemorySearchPreviewTool` to an agent registered in Foundry
- Running conversations via `FoundryAgent` from `agent_framework.foundry`
- Per-user memory scoping with `scope`
- Using low-level memory APIs to search and update memories directly
- Cross-session memory persistence

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os, time
from pathlib import Path
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
load_dotenv()

from utils.config import get_env, get_credential

# azure.ai.projects — for memory store management (create/delete/update/search)
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    MemorySearchPreviewTool,
    MemoryStoreDefaultDefinition,
    MemoryStoreDefaultOptions,
    MemorySearchOptions,
)

# agent_framework.foundry — for running agent conversations
from agent_framework.foundry import FoundryAgent

# Endpoints
#PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")
PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_OTHER")
MODEL = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")
EMBEDDING_MODEL = get_env("AZURE_AI_EMBEDDING_MODEL", required=False) or "text-embedding-ada-002"

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=get_credential(),
)

print(f"📡 Project endpoint: {PROJECT_ENDPOINT}")
print(f"🧠 Chat model: {MODEL}")
print(f"📐 Embedding model: {EMBEDDING_MODEL}")
print("✅ Connected to Foundry")

c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


📡 Project endpoint: https://foundry-demo-ncus-resource.services.ai.azure.com/api/projects/prj-foundry-demo-ncus-dev-001
🧠 Chat model: gpt-4o
📐 Embedding model: text-embedding-ada-002
✅ Connected to Foundry


---
### Step 1 — Create a Memory Store 🗄️

A memory store is the persistent backend that holds extracted user preferences. We configure it with:
- A **chat model** for memory extraction and summarization
- An **embedding model** for semantic search over memories
- `user_profile_details` to guide *what* to remember (and what to skip)

In [3]:
# ── 1. Create the Memory Store (via azure.ai.projects) ─────────────────────────
MEMORY_STORE_NAME = "shopping-assistant-memory"

# Configure what the memory system should extract and store
options = MemoryStoreDefaultOptions(
    chat_summary_enabled=True,
    user_profile_enabled=True,
    user_profile_details=(
        "Store clothing sizes, style preferences, favorite brands, color preferences, "
        "and budget constraints. Avoid storing payment details, addresses, or other "
        "personally identifiable information."
    ),
)

definition = MemoryStoreDefaultDefinition(
    chat_model=MODEL,
    embedding_model=EMBEDDING_MODEL,
    options=options,
)

# Create (or retrieve if it already exists)
try:
    memory_store = project.beta.memory_stores.create(
        name=MEMORY_STORE_NAME,
        definition=definition,
        description="Persistent memory for the personal shopping assistant",
    )
    print(f"✅ Created memory store: {memory_store.name}")
except Exception as e:
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        print(f"ℹ️  Memory store '{MEMORY_STORE_NAME}' already exists — reusing it.")
    else:
        raise

ℹ️  Memory store 'shopping-assistant-memory' already exists — reusing it.


---
### Step 2 — Register Agent with Memory & Connect via FoundryAgent 🛍️

We register the agent in Foundry with the `MemorySearchPreviewTool` attached (via `azure.ai.projects`), then connect via `FoundryAgent` from the Agent Framework package to run conversations.

The memory tool:
- Automatically **reads** relevant memories at the start of each conversation
- Automatically **writes** new preferences after the conversation settles

In [4]:
# ── 2. Register agent in Foundry, connect via FoundryAgent ────────────────────
AGENT_NAME = "PersonalShopper-af"
USER_SCOPE = "demo_user_jane"  # Per-user memory isolation

# Memory search tool — agent reads/writes memories automatically
memory_tool = MemorySearchPreviewTool(
    memory_store_name=MEMORY_STORE_NAME,
    scope=USER_SCOPE,
    update_delay=1,  # 1 second for demo; use 300 (5 min) in production
)

# Register agent version in Foundry (via azure.ai.projects)
agent_version = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a personal shopping assistant. You help users find clothing "
            "and accessories that match their style. You remember their preferences "
            "from previous conversations — sizes, favorite brands, colors, and budget. "
            "When making recommendations, always consider what you know about the user. "
            "If you don't know something, ask."
        ),
        tools=[memory_tool],
    ),
)

# Connect via Agent Framework's FoundryAgent for running conversations
shopping_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=agent_version.name,
    agent_version=str(agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {agent_version.name} v{agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(shopping_agent).__name__}")
print(f"🔑 Memory scoped to: {USER_SCOPE}")

2026-10-26 11:56:32,617 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
✅ Agent registered in Foundry: PersonalShopper-af v1
🤖 FoundryAgent connected: FoundryAgent
🔑 Memory scoped to: demo_user_jane


---
### Step 3 — First Conversation: Teach the Agent Your Preferences 💬

In this first session, the user shares their preferences naturally via `FoundryAgent.run()`. The memory store extracts and persists them in the background.

In [5]:
# ── 3. First conversation — user shares preferences (via FoundryAgent) ────────
messages_session_1 = [
    "Hi! I'm looking for some new clothes. I usually wear size M for tops and 32 for pants.",
    "I love minimalist Scandinavian style — brands like COS, Arket, and Uniqlo. Mostly neutrals: black, white, grey, and navy.",
    "My budget is usually $50-150 per item. I don't mind spending more on outerwear though, up to $300.",
]

print("📝 Session 1 — Teaching preferences\n")
for msg in messages_session_1:
    print(f"👤 User: {msg}")
    result = await shopping_agent.run(msg)
    print(f"🤖 Agent: {result}\n")

print("\n⏳ Waiting for memories to be extracted and stored...")
time.sleep(65)  # Memory extraction happens after conversation inactivity
print("✅ Memories should now be stored.")

📝 Session 1 — Teaching preferences

👤 User: Hi! I'm looking for some new clothes. I usually wear size M for tops and 32 for pants.
🤖 Agent: Got it! You wear size M for tops and 32 for pants. What kind of clothes are you looking for today? Casual, formal, or something specific?

👤 User: I love minimalist Scandinavian style — brands like COS, Arket, and Uniqlo. Mostly neutrals: black, white, grey, and navy.
🤖 Agent: Got it! You love minimalist Scandinavian style, brands like COS, Arket, and Uniqlo, and neutral colors like black, white, grey, and navy. Let me know if you're looking for something specific, and I'll find the perfect recommendations for you!

👤 User: My budget is usually $50-150 per item. I don't mind spending more on outerwear though, up to $300.
🤖 Agent: Got it! Your budget is $50-150 per item, except for outerwear where you're comfortable spending up to $300. Let me know what you're looking for, and I’ll help you find the perfect piece!


⏳ Waiting for memories to be extr

---
### Step 4 — Verify Stored Memories 🔍

Use the low-level memory API (via `azure.ai.projects`) to inspect what was persisted. This is useful for debugging and auditing.

In [6]:
# ── 4. Search stored memories (via azure.ai.projects) ─────────────────────────
query = {"role": "user", "content": "What are my clothing preferences?", "type": "message"}

search_response = project.beta.memory_stores.search_memories(
    name=MEMORY_STORE_NAME,
    scope=USER_SCOPE,
    items=[query],
    options=MemorySearchOptions(max_memories=10),
)

print(f"🧠 Found {len(search_response.memories)} stored memories:\n")
for mem in search_response.memories:
    print(f"  • {mem.memory_item.content}")

🧠 Found 6 stored memories:

  • The user expressed a preference for minimalist Scandinavian style and mentioned liking brands such as COS, Arket, and Uniqlo. They also stated a preference for clothing in neutral colors, including black, white, grey, and navy.
  • The user mentioned they are looking to purchase new clothes. They shared that they typically wear a size M for tops and size 32 for pants.
  • The user mentioned their usual budget for shopping, which is $50 to $150 per item. They specified that they are willing to spend more on outerwear, with a maximum budget of up to $300.
  • User wears size M for tops and 32 for pants. User is currently looking to buy clothes.
  • User loves minimalist Scandinavian style and prefers brands like COS, Arket, and Uniqlo. They mainly wear neutral colors such as black, white, grey, and navy.
  • User's budget per item is $50-150 under normal circumstances, with up to $300 especially for outerwear.


In [7]:
# ── 4b. Show memories by type (user_profile vs chat_summary) ──────────────────
search_response = project.beta.memory_stores.search_memories(
    name=MEMORY_STORE_NAME,
    scope=USER_SCOPE,
    items=[{"role": "user", "content": "What did we discuss and what are my preferences?", "type": "message"}],
    options=MemorySearchOptions(max_memories=10),
)

user_profile_memories = [m for m in search_response.memories if m.memory_item.kind == "user_profile"]
chat_summary_memories = [m for m in search_response.memories if m.memory_item.kind == "chat_summary"]

print(f"📋 User Profile Memories ({len(user_profile_memories)}):")
print("   (Structured facts extracted from conversations)\n")
for mem in user_profile_memories:
    print(f"  • {mem.memory_item.content}")

print(f"\n💬 Chat Summary Memories ({len(chat_summary_memories)}):")
print("   (Auto-generated summaries of past conversations)\n")
for mem in chat_summary_memories:
    print(f"  • {mem.memory_item.content}")

📋 User Profile Memories (3):
   (Structured facts extracted from conversations)

  • User loves minimalist Scandinavian style and prefers brands like COS, Arket, and Uniqlo. They mainly wear neutral colors such as black, white, grey, and navy.
  • User wears size M for tops and 32 for pants. User is currently looking to buy clothes.
  • User's budget per item is $50-150 under normal circumstances, with up to $300 especially for outerwear.

💬 Chat Summary Memories (3):
   (Auto-generated summaries of past conversations)

  • The user mentioned they are looking to purchase new clothes. They shared that they typically wear a size M for tops and size 32 for pants.
  • The user expressed a preference for minimalist Scandinavian style and mentioned liking brands such as COS, Arket, and Uniqlo. They also stated a preference for clothing in neutral colors, including black, white, grey, and navy.
  • The user mentioned their usual budget for shopping, which is $50 to $150 per item. They specifi

---
### Step 5 — New Conversation: Agent Recalls Preferences Automatically 🔄

Start a completely new conversation via `FoundryAgent`. The agent should recall stored preferences without being told again — demonstrating cross-session persistence.

In [8]:
# ── 5. Second conversation — agent uses stored memory (via FoundryAgent) ──────
messages_session_2 = [
    "Hey, can you recommend a good jacket for autumn?",
    "What about a casual shirt I can wear to the office?",
]

print("📝 Session 2 — Agent recalls preferences from memory\n")
for msg in messages_session_2:
    print(f"👤 User: {msg}")
    result = await shopping_agent.run(msg)
    print(f"🤖 Agent: {result}\n")

print("💡 Notice the agent uses your size (M), style (minimalist), brands (COS, Arket),")
print("   colors (neutrals), and budget ($50-150, up to $300 for outerwear) — all from memory!")

📝 Session 2 — Agent recalls preferences from memory

👤 User: Hey, can you recommend a good jacket for autumn?
🤖 Agent: For an autumn jacket that aligns with your minimalist Scandinavian aesthetic and neutral color preferences, here are a few recommendations within your budget and size range:

### 1. **Arket Wool Blend Overshirt**  
   - **Description**: A lightly structured overshirt made from a wool blend, perfect for layering. It has a clean, tailored design, which makes it versatile for both casual and semi-formal settings.  
   - **Colors**: Available in grey, navy, or black.  
   - **Price**: Around $200.  
   - **Size**: M  

### 2. **COS Relaxed-Fit Wool Jacket**  
   - **Description**: A contemporary take on a classic silhouette with dropped shoulders for a relaxed fit. Made from responsibly sourced wool.  
   - **Colors**: Charcoal grey or black.  
   - **Price**: $250.  
   - **Size**: Medium fits perfectly for layering.  

### 3. **Uniqlo Hybrid Down Parka**  
   - **Descrip

---
### Step 6 — Update Preferences (Memory Evolves) 🔄

User preferences change over time. The memory store consolidates updates — it doesn't just append, it merges and deduplicates.

In [9]:
# ── 6. Third conversation — update preferences (via FoundryAgent) ─────────────
update_messages = [
    "Actually, I've been getting into earth tones lately — olive green, rust, and tan.",
    "Also I just sized up to L for tops, so keep that in mind for future recommendations.",
]

print("📝 Session 3 — Updating preferences\n")
for msg in update_messages:
    print(f"👤 User: {msg}")
    result = await shopping_agent.run(msg)
    print(f"🤖 Agent: {result}\n")

print("⏳ Waiting for memory update...")
time.sleep(65)
print("✅ Memory updated.")

📝 Session 3 — Updating preferences

👤 User: Actually, I've been getting into earth tones lately — olive green, rust, and tan.
🤖 Agent: Got it! I'll keep in mind your new interest in earth tones like olive green, rust, and tan for future recommendations. Are you looking for anything specific today, like tops, pants, or accessories?

👤 User: Also I just sized up to L for tops, so keep that in mind for future recommendations.
🤖 Agent: Got it! I'll keep in mind that you've sized up to L for tops when recommending clothing in the future.

⏳ Waiting for memory update...
✅ Memory updated.


In [10]:
# ── 6b. Verify updated memories (via azure.ai.projects) ──────────────────────
search_response = project.beta.memory_stores.search_memories(
    name=MEMORY_STORE_NAME,
    scope=USER_SCOPE,
    items=[{"role": "user", "content": "What size and colors do I prefer?", "type": "message"}],
    options=MemorySearchOptions(max_memories=10),
)

print(f"🧠 Updated memories ({len(search_response.memories)} items):\n")
for mem in search_response.memories:
    print(f"  • {mem.memory_item.content}")

print("\n💡 Notice size updated from M → L and new earth tone colors added.")

🧠 Updated memories (10 items):

  • The user expressed a preference for minimalist Scandinavian style and mentioned liking brands such as COS, Arket, and Uniqlo. They also stated a preference for clothing in neutral colors, including black, white, grey, and navy.
  • User wears size L for tops and 32 for pants. User is currently looking to buy clothes, including a casual shirt appropriate for office settings. User's clothing size is L, and they prefer size L for tops to be considered in future clothing recommendations.
  • User loves minimalist Scandinavian style and prefers brands like COS, Arket, and Uniqlo. They mainly wear neutral colors such as black, white, grey, and navy, and have recently developed an interest in earth tones such as olive green, rust, and tan. User has a minimalist Scandinavian aesthetic and prefers neutral color palettes.
  • The user mentioned they are looking to purchase new clothes. They shared that they typically wear a size M for tops and size 32 for pant

---
### Step 7 — Direct Memory API: Add Memories Programmatically 📥

You can also write memories directly via the API — useful for importing user profiles from other systems or seeding initial context.

In [15]:
# ── 7. Add memory via low-level API (via azure.ai.projects) ───────────────────
# Simulate importing a preference from an external system (e.g., past purchase data)
import_message = {
    "role": "user",
    "content": "I bought a navy wool overcoat from COS last winter and loved it. I'm also allergic to synthetic fabrics.",
    "type": "message",
}

update_poller = project.beta.memory_stores.begin_update_memories(
    name=MEMORY_STORE_NAME,
    scope=USER_SCOPE,
    items=[import_message],
    update_delay=0,  # Process immediately
)

update_result = update_poller.result()
print(f"✅ Added {len(update_result.memory_operations)} memory operation(s):")
for op in update_result.memory_operations:
    print(f"  • [{op.kind}] {op.memory_item.content}")

✅ Added 2 memory operation(s):
  • [created] User is allergic to synthetic fabrics.
  • [updated] The user loves minimalist Scandinavian style with brands like COS, Arket, and Uniqlo, and has a preference for neutral colors: black, white, grey, and navy. The user has started to get into earth tones like olive green, rust, and tan, showing a growing preference for these tones. They bought a navy wool overcoat from COS in the winter of 2025 and liked it.


---
### Step 8 — Final Test: Full Preference Recall 🎯

Verify the agent now knows the complete, updated user profile including the programmatically added memory.

In [11]:
# ── 8. Final conversation — full preference recall (via FoundryAgent) ─────────
query = (
    "I need a complete outfit for a casual Friday at work. "
    "What do you suggest based on everything you know about me?"
)

print(f"👤 User: {query}\n")
result = await shopping_agent.run(query)
print(f"🤖 Agent: {result}")

print("\n💡 The agent should reference: size L, earth tones + neutrals, COS/Arket/Uniqlo,")
print("   $50-150 budget, natural fabrics (no synthetics), and minimalist style.")

👤 User: I need a complete outfit for a casual Friday at work. What do you suggest based on everything you know about me?

🤖 Agent: Thanks for sharing your preferences! Based on what I know, I'd recommend a polished yet comfortable outfit for Casual Friday at work. Here's a complete look within your preferred minimalist Scandinavian aesthetic and neutral color palette—while also staying within your budget:

---

### **Top:**
- **Arket Classic Oxford Shirt in Grey Melange ($89)**  
  A timeless and versatile shirt that offers a polished yet casual look suitable for the office.

---

### **Bottoms:**
- **COS Slim-Fit Trousers in Navy ($120)**  
  These tailored pants provide a clean silhouette, pairing seamlessly with the shirt for a smart-casual vibe.

---

### **Outerwear:**
- **Arket Wool Blend Overshirt in Black ($200)**  
  A structured and lightweight jacket perfect for autumn weather. Adds a polished layer while keeping the look minimalist and work-appropriate.

---

### **Shoes:**

---
### Cleanup 🧹

Remove the demo agent and memory store.

In [12]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
# Delete agent from Foundry
try:
    project.agents.delete(agent_name=AGENT_NAME)
    print(f"🗑️  Deleted agent: {AGENT_NAME}")
except Exception as e:
    print(f"⚠️  Could not delete agent: {e}")

# Delete memory scope (user's data)
try:
    project.beta.memory_stores.delete_scope(
        name=MEMORY_STORE_NAME,
        scope=USER_SCOPE,
    )
    print(f"🗑️  Deleted memories for scope: {USER_SCOPE}")
except Exception as e:
    print(f"⚠️  Could not delete scope: {e}")

# Delete memory store
try:
    project.beta.memory_stores.delete(MEMORY_STORE_NAME)
    print(f"🗑️  Deleted memory store: {MEMORY_STORE_NAME}")
except Exception as e:
    print(f"⚠️  Could not delete memory store: {e}")

print("\n✅ Cleanup complete!")

🗑️  Deleted agent: PersonalShopper-af
🗑️  Deleted memories for scope: demo_user_jane
🗑️  Deleted memory store: shopping-assistant-memory

✅ Cleanup complete!
